In [0]:
# from pyspark.sql import functions as F
# from datetime import datetime, timezone

CATALOG        = "main"
DATABASE       = "flight_tracker"
VOLUME_ROOT    = f"/Volumes/{CATALOG}/{DATABASE}/raw_files"
REGIONS = {
    "hyderabad": f"{VOLUME_ROOT}/hyderabad",
    "mumbai":    f"{VOLUME_ROOT}/mumbai",
    "delhi":     f"{VOLUME_ROOT}/delhi",
}
BRONZE_TABLE   = f"{CATALOG}.{DATABASE}.flight_raw"

print(f"Catalog        : {CATALOG}")
print(f"Database       : {DATABASE}")
print(f"Volume Root    : {VOLUME_ROOT}")
print(f"Bronze Table   : {BRONZE_TABLE}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{DATABASE}")
print(f"✅ Schema {CATALOG}.{DATABASE} ready")

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{DATABASE}.raw_files
    COMMENT 'Raw ADS-B JSON files uploaded from GitHub extraction'
""")
print(f"✅ Volume {CATALOG}.{DATABASE}.raw_files ready")
print(f"   Upload your JSON files to: {VOLUME_ROOT}/hyderabad/  etc.")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    ingestion_id     STRING     COMMENT 'Unique UUID per row',
    fetched_at       TIMESTAMP  COMMENT 'UTC timestamp of API fetch',
    source_region    STRING     COMMENT 'Region: hyderabad / mumbai / delhi',
    hex              STRING     COMMENT 'ICAO 24-bit aircraft address',
    flight           STRING     COMMENT 'Raw callsign from API',
    lat              DOUBLE,
    lon              DOUBLE,
    alt_baro         INT        COMMENT 'Barometric altitude (ft) — ground=NULL',
    gs               DOUBLE     COMMENT 'Ground speed (knots)',
    track            DOUBLE     COMMENT 'Heading (degrees)',
    baro_rate        INT        COMMENT 'Vertical rate (ft/min)',
    squawk           STRING,
    emergency        STRING,
    registration     STRING     COMMENT 'Tail number',
    aircraft_type    STRING     COMMENT 'ICAO type code e.g. B77W',
    category         STRING,
    seen             DOUBLE,
    db_flags         INT        COMMENT 'Bitmask: bit0=military',
    raw_payload      STRING     COMMENT 'Full original JSON — never modified',
    _ingested_at     TIMESTAMP  COMMENT 'Databricks ingest timestamp',
    _source_file     STRING     COMMENT 'Source file path'
)
USING DELTA
COMMENT 'Bronze: append-only raw ADS-B flight observations'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
)
""")
print(f"✅ Table {BRONZE_TABLE} ready")


In [0]:
%pip install requests



In [0]:
# Databricks notebook source

# COMMAND ----------
# MAGIC %md
# MAGIC # 🟫 BRONZE LAYER — Fetch from ADSB.lol API → Save to Volume → Load to Delta

# COMMAND ----------
# Step 1: Install libraries

# COMMAND ----------
# Step 2: Imports and config

import requests, json, os, uuid, time
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

CATALOG      = "main"
DATABASE     = "flight_tracker"
BRONZE_TABLE = f"{CATALOG}.{DATABASE}.flight_raw"
VOLUME_PATH  = f"/Volumes/{CATALOG}/{DATABASE}/raw_json"
API_BASE     = "https://api.adsb.lol"

# Region name, latitude, longitude, radius in nautical miles
REGIONS = [
    ("hyderabad", 17.3850, 78.4867, 250),
    ("mumbai",    19.0760, 72.8777, 250),
    ("delhi",     28.6139, 77.2090, 250),
]

print(f"Bronze Table : {BRONZE_TABLE}")
print(f"Volume Path  : {VOLUME_PATH}")

# COMMAND ----------
# Step 3: Create schema, volume, and bronze table (run once)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{DATABASE}")

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{DATABASE}.raw_json
    COMMENT 'Raw JSON snapshots from ADSB.lol API'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    ingestion_id   STRING,
    fetched_at     TIMESTAMP,
    source_region  STRING,
    api_endpoint   STRING,
    hex            STRING,
    flight         STRING,
    lat            DOUBLE,
    lon            DOUBLE,
    alt_baro       INT,
    alt_geom       INT,
    gs             DOUBLE,
    track          DOUBLE,
    baro_rate      INT,
    squawk         STRING,
    emergency      STRING,
    category       STRING,
    registration   STRING,
    aircraft_type  STRING,
    ownop          STRING,
    db_flags       INT,
    nav_modes      STRING,
    mlat           STRING,
    seen           DOUBLE,
    rssi           DOUBLE,
    raw_payload    STRING,
    _ingested_at   TIMESTAMP,
    _source_file   STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
)
""")

print(f"✅ Schema, Volume, and Table ready")

# COMMAND ----------
# Step 4: Fetch from API and save raw JSON to Volume

def fetch_and_save(region, lat, lon, dist_nm):
    """
    Calls ADSB.lol API and saves the raw JSON response to the Volume.
    Returns (payload_dict, saved_file_path).
    """
    url = f"{API_BASE}/v2/lat/{lat}/lon/{lon}/dist/{dist_nm}"
    print(f"\n  Calling API: {url}")

    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    payload = resp.json()

    aircraft_count = payload.get("total", len(payload.get("ac", [])))
    print(f"  Aircraft returned: {aircraft_count}")

    # Save raw JSON to Volume
    now_str   = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S")
    file_path = f"{VOLUME_PATH}/{region}/{now_str}.json"
    os.makedirs(f"{VOLUME_PATH}/{region}", exist_ok=True)

    with open(file_path, "w") as f:
        json.dump(payload, f)

    print(f"  Saved to Volume: {file_path}")
    return payload, file_path


# Run for all regions
raw_results = []
for region_name, lat, lon, dist_nm in REGIONS:
    try:
        payload, file_path = fetch_and_save(region_name, lat, lon, dist_nm)
        raw_results.append((region_name, payload, file_path,
                            f"{API_BASE}/v2/lat/{lat}/lon/{lon}/dist/{dist_nm}"))
    except Exception as e:
        print(f"  ❌ {region_name} failed: {e}")

print(f"\n✅ API fetch done — {len(raw_results)} regions fetched")

# COMMAND ----------
# Step 5: Parse API responses and load into Bronze Delta table

# Define explicit schema matching the existing Bronze table (created from Cell 2)
bronze_schema = StructType([
    StructField("ingestion_id", StringType(), True),
    StructField("fetched_at", StringType(), True),
    StructField("source_region", StringType(), True),
    StructField("hex", StringType(), True),
    StructField("flight", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("alt_baro", IntegerType(), True),
    StructField("gs", DoubleType(), True),
    StructField("track", DoubleType(), True),
    StructField("baro_rate", IntegerType(), True),
    StructField("squawk", StringType(), True),
    StructField("emergency", StringType(), True),
    StructField("registration", StringType(), True),
    StructField("aircraft_type", StringType(), True),
    StructField("category", StringType(), True),
    StructField("seen", DoubleType(), True),
    StructField("db_flags", IntegerType(), True),
    StructField("raw_payload", StringType(), True),
    StructField("_ingested_at", StringType(), True),
    StructField("_source_file", StringType(), True),
])

def parse_and_load(region, payload, file_path, api_url):
    """
    Parse the API JSON response and append rows to the Bronze Delta table.
    Returns number of rows written.
    """
    aircraft_list = payload.get("ac", [])
    if not aircraft_list:
        print(f"  ⚠️  No aircraft for {region}")
        return 0

    # Convert epoch timestamp from API (API returns milliseconds, convert to seconds)
    api_now = payload.get("now")
    fetched_at = (
        datetime.fromtimestamp(api_now / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
        if api_now else
        datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    )
    ingested_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

    rows = []
    for ac in aircraft_list:
        if not ac.get("hex"):
            continue

        def si(v):   # safe int
            try: return int(v) if v is not None else None
            except: return None

        def sf(v):   # safe float
            try: return float(v) if v is not None else None
            except: return None

        def ss(v):   # safe string (converts list/dict to JSON)
            if v is None: return None
            return json.dumps(v) if isinstance(v, (list, dict)) else str(v)

        rows.append({
            "ingestion_id":  str(uuid.uuid4()),
            "fetched_at":    fetched_at,
            "source_region": region,
            "hex":           ss(ac.get("hex")),
            "flight":        ss(ac.get("flight")),
            "lat":           sf(ac.get("lat")),
            "lon":           sf(ac.get("lon")),
            "alt_baro":      si(ac.get("alt_baro")) if ac.get("alt_baro") != "ground" else None,
            "gs":            sf(ac.get("gs")),
            "track":         sf(ac.get("track")),
            "baro_rate":     si(ac.get("baro_rate")),
            "squawk":        ss(ac.get("squawk")),
            "emergency":     ss(ac.get("emergency")),
            "registration":  ss(ac.get("r")),
            "aircraft_type": ss(ac.get("t")),
            "category":      ss(ac.get("category")),
            "seen":          sf(ac.get("seen")),
            "db_flags":      si(ac.get("dbFlags")),
            "raw_payload":   json.dumps(ac),
            "_ingested_at":  ingested_at,
            "_source_file":  file_path,
        })

    df = spark.createDataFrame(rows, schema=bronze_schema)
    df = df.withColumns({
        "fetched_at": F.to_timestamp("fetched_at"),
        "_ingested_at": F.to_timestamp("_ingested_at")
    })
    df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
    print(f"  ✅ {region}: {len(rows):,} rows written to Bronze")
    return len(rows)


# Load all fetched regions
total = 0
for region_name, payload, file_path, api_url in raw_results:
    total += parse_and_load(region_name, payload, file_path, api_url)

print(f"\n✅ Total rows written to Bronze: {total:,}")

# COMMAND ----------
# Step 6: Verify

print("=== Row count ===")
spark.sql(f"SELECT COUNT(*) AS total FROM {BRONZE_TABLE}").show()

print("=== Rows per region ===")
spark.sql(f"""
    SELECT source_region, COUNT(*) AS rows, COUNT(DISTINCT hex) AS aircraft,
           MAX(fetched_at) AS latest
    FROM {BRONZE_TABLE}
    GROUP BY source_region
""").show()

print("=== Sample rows ===")
spark.sql(f"""
    SELECT fetched_at, source_region, hex, flight, lat, lon,
           alt_baro, gs, registration, aircraft_type, category
    FROM {BRONZE_TABLE}
    ORDER BY fetched_at DESC
    LIMIT 10
""").show(truncate=False)

print("=== Files saved in Volume ===")
for region_name, _, _, _ in raw_results:
    try:
        files = dbutils.fs.ls(f"{VOLUME_PATH}/{region_name}")
        for f in files:
            print(f"  {f.path}  ({f.size} bytes)")
    except:
        print(f"  No files yet for {region_name}")
        

In [0]:
CATALOG = "main"
DATABASE = "flight_tracker"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{DATABASE}/raw_files"

try:
    files = dbutils.fs.ls(VOLUME_ROOT)
    print(f"Files found in volume:")
    for f in files:
        print(f"  {f.path}  ({f.size} bytes)")
except Exception as e:
    print(f"Volume is empty or not yet populated: {e}")
    print(f"\n👉 Upload your JSON files to: {VOLUME_ROOT}/hyderabad/")

In [0]:
def load_bronze_from_volume(json_dir: str, source_region: str):
    try:
        file_list = dbutils.fs.ls(json_dir)
        json_files = [f.path for f in file_list if f.name.endswith(".json")]
        if not json_files:
            print(f"⚠️  No .json files found in {json_dir} — skipping {source_region}")
            return 0
        print(f"   Found {len(json_files)} JSON files in {json_dir}")
    except Exception:
        print(f"⚠️  Folder not found: {json_dir} — skipping {source_region}")
        return 0

    raw_df = (
        spark.read
             .option("multiline", "true")
             .json(json_dir)
    )

    exploded = raw_df.select(
        F.col("now").cast("timestamp").alias("fetched_at"),
        F.input_file_name().alias("_source_file"),
        F.explode(F.col("ac")).alias("ac")
    )

    bronze_df = exploded.select(
        F.expr("uuid()").alias("ingestion_id"),
        F.col("fetched_at"),
        F.lit(source_region).alias("source_region"),
        F.col("ac.hex").alias("hex"),
        F.trim(F.col("ac.flight")).alias("flight"),
        F.col("ac.lat").cast("double").alias("lat"),
        F.col("ac.lon").cast("double").alias("lon"),
        F.when(
            F.col("ac.alt_baro").cast("string").rlike(r"^-?\d+$"),
            F.col("ac.alt_baro").cast("int")
        ).otherwise(F.lit(None).cast("int")).alias("alt_baro"),
        F.col("ac.gs").cast("double").alias("gs"),
        F.col("ac.track").cast("double").alias("track"),
        F.col("ac.baro_rate").cast("int").alias("baro_rate"),
        F.col("ac.squawk").alias("squawk"),
        F.col("ac.emergency").alias("emergency"),
        F.col("ac.r").alias("registration"),
        F.col("ac.t").alias("aircraft_type"),
        F.col("ac.category").alias("category"),
        F.col("ac.seen").cast("double").alias("seen"),
        F.col("ac.dbFlags").cast("int").alias("db_flags"),
        F.to_json(F.col("ac")).alias("raw_payload"),
        F.current_timestamp().alias("_ingested_at"),
        F.col("_source_file"),
    )

    count = bronze_df.count()
    bronze_df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
    print(f"✅ {source_region}: appended {count:,} rows")
    return count


In [0]:
total = 0
for region, path in REGIONS.items():
    total += load_bronze_from_volume(path, region)

print(f"\n🏁 Total rows ingested to Bronze: {total:,}")

print("=== Row Count ===")
display(spark.sql(f"SELECT COUNT(*) AS total_rows FROM {BRONZE_TABLE}"))

print("=== Rows per Region ===")
display(spark.sql(f"""
    SELECT source_region,
           COUNT(*)         AS row_count,
           MIN(fetched_at)  AS earliest,
           MAX(fetched_at)  AS latest
    FROM {BRONZE_TABLE}
    GROUP BY source_region
    ORDER BY row_count DESC
"""))

print("=== Sample Rows ===")
display(spark.sql(f"""
    SELECT ingestion_id, fetched_at, source_region, hex, flight,
           lat, lon, alt_baro, gs, registration, aircraft_type
    FROM {BRONZE_TABLE}
    ORDER BY fetched_at DESC
    LIMIT 10
"""))

print("=== Null Check ===")
display(spark.sql(f"""
    SELECT COUNT(*)                                              AS total,
           SUM(CASE WHEN hex IS NULL    THEN 1 ELSE 0 END)      AS null_hex,
           SUM(CASE WHEN lat IS NULL    THEN 1 ELSE 0 END)      AS null_lat,
           SUM(CASE WHEN lon IS NULL    THEN 1 ELSE 0 END)      AS null_lon,
           SUM(CASE WHEN alt_baro IS NULL THEN 1 ELSE 0 END)    AS null_alt_baro
    FROM {BRONZE_TABLE}
"""))

print("=== Delta History ===")
display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").select(
    "version","timestamp","operation","operationMetrics"
).limit(5))

In [0]:
spark.sql(f"""
SELECT
    MIN(fetched_at),
    MAX(fetched_at)
FROM {BRONZE_TABLE}
""").show(truncate=False)